In [2]:
import xarray as xr
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
import matplotlib.pyplot as plt
import seaborn as sns

### Load SPEI Data
Will load the NetCDF data file for SPEI and extract the time, latitude, longitude and SPEI values.

In [3]:
# === Load SPEI data ===
spei_nc_path = "../data/raw/spei03.nc"

# Load NetCDF data
ds = xr.open_dataset(spei_nc_path)
spei = ds['spei']
lats = ds['lat'].values
lons = ds['lon'].values
times = pd.to_datetime(ds['time'].values)

# Print basic information about the data
print("Latitude range:", lats.min(), "-", lats.max())
print("Longitude range:", lons.min(), "-", lons.max())
print("Time range:", times.min(), "-", times.max())

Latitude range: -89.75 - 89.75
Longitude range: -179.75 - 179.75
Time range: 1901-01-16 00:00:00 - 2023-12-16 00:00:00


### Read the country boundary vector file
Will load the shapefile file for the country boundaries and extract the country names and ISO3 codes.

In [4]:
# === Read country border vector files ===
shapefile_path = '../data/raw/ne_50m_admin_0_countries/ne_50m_admin_0_countries.shp'
countries = gpd.read_file(shapefile_path).to_crs('EPSG:4326')

# Extract country names and ISO3 codes
namelist = countries[['ADMIN', 'ISO_A3']].drop_duplicates().reset_index(drop=True)
namelist = namelist.rename(columns={'ADMIN': 'country_name', 'ISO_A3': 'iso3'})

# Save to processed folder
namelist.to_csv('../data/processed/ne_50m_admin_0_countries_namelist.csv', index=False)
print("Country names and ISO3 codes have been saved to: ne_50m_admin_0_countries_namelist.csv")

Country names and ISO3 codes have been saved to: ne_50m_admin_0_countries_namelist.csv


ne_50m_admin_0_countries_namelist.csv contains 242 countries, but there are some problems with the quality of this data, with some countries having an iso3 code of -99.

### Construct SPEI grid points GeoDataFrame
Converts latitude/longitude grid points from SPEI data into a GeoDataFrame for spatial connectivity.

In [5]:
# === Constructing SPEI Grid Points GeoDataFrame ===
lon_grid, lat_grid = np.meshgrid(lons, lats)
points = [Point(lon, lat) for lon, lat in zip(lon_grid.flatten(), lat_grid.flatten())]
points_gdf = gpd.GeoDataFrame({'orig_idx': np.arange(len(points))}, geometry=points, crs='EPSG:4326')

### Spatial connectivity
Assign each grid point to the corresponding country.

In [6]:
# === space connection ===
points_gdf = gpd.sjoin(points_gdf, countries[['ADMIN', 'ISO_A3', 'geometry']], how='inner', predicate='within')
points_gdf = points_gdf.rename(columns={'ADMIN': 'country', 'ISO_A3': 'iso3'})
print("Space connection completed, sample data:")
points_gdf.head()

Space connection completed, sample data:


,orig_idx,geometry,index_right,country,iso3
0,0,POINT (-179.75 -89.75),239,Antarctica,ATA
1,1,POINT (-179.25 -89.75),239,Antarctica,ATA
2,2,POINT (-178.75 -89.75),239,Antarctica,ATA
3,3,POINT (-178.25 -89.75),239,Antarctica,ATA
4,4,POINT (-177.75 -89.75),239,Antarctica,ATA


### Extract monthly SPEI averages per country
Calculate the average monthly SPEI per country for 2019-2022.

In [7]:
# === Monthly SPEI averages per country extracted and lagged ===
records = []
for t_idx, t in enumerate(times):
    if t.year < 2018 or t.year > 2022:  # Extract time range for lagged processing (including 2018 for lag calculation)
        continue
    spei_slice = spei.isel(time=t_idx).values.flatten()
    points_gdf['spei'] = spei_slice[points_gdf['orig_idx'].values]
    grouped = points_gdf.groupby(['country', 'iso3'])['spei'].mean().reset_index()
    grouped['spei_date'] = t
    records.append(grouped)

# Merge all time points' SPEI data
result = pd.concat(records, ignore_index=True)
result = result.dropna(subset=['country'])

# === lagging process ===
# Split the date into year and month
result['year'] = result['spei_date'].dt.year
result['month'] = result['spei_date'].dt.month

# Create lagged columns
for lag in [1, 2, 3]:  # Lag 1 month, 2 months, 3 months
    result[f'spei_lag{lag}'] = result.groupby('iso3')['spei'].shift(lag)

# Filter final time range (2019.1 - 2022.12)
result = result[(result['year'] >= 2019) & (result['year'] <= 2022)]

In [8]:
# Check data after lag processing
print("Data after lag processing:")
result

Data after lag processing:


,country,iso3,spei,spei_date,year,month,spei_lag1,spei_lag2,spei_lag3
2280,Afghanistan,AFG,-0.101823,2019-01-16,2019,1,-0.092170,0.173375,-0.499876
2281,Aland,ALA,-0.584555,2019-01-16,2019,1,-0.591903,-0.573946,-0.568361
2282,Albania,ALB,-0.229678,2019-01-16,2019,1,-1.489391,-1.807115,-1.576336
2283,Algeria,DZA,-1.081419,2019-01-16,2019,1,-0.542777,0.313114,0.784864
2284,Angola,AGO,0.090749,2019-01-16,2019,1,0.083615,0.132651,0.015039
...,...,...,...,...,...,...,...,...,...
11395,Western Sahara,ESH,-1.458937,2022-12-16,2022,12,-0.939643,-0.582081,-0.597001
11396,Yemen,YEM,-1.207492,2022-12-16,2022,12,-0.505910,0.006843,0.205940
11397,Zambia,ZMB,0.084212,2022-12-16,2022,12,0.138420,0.039404,-0.045226
11398,Zimbabwe,ZWE,-0.164138,2022-12-16,2022,12,-0.331173,-0.593602,-0.502948


In [10]:
result.to_csv('../data/processed/spei_result_unclean.csv', index=False)
print("The preview of the result has been saved to the spei_result_unclean.csv")

The preview of the result has been saved to the spei_result_unclean.csv


### Data checking
Checks the extracted SPEI data for completeness and corrects possible errors.

In [11]:
# 1. shp file of all countries/regions
all_admin = set(countries['ADMIN'].unique())
print(f"Number of unique ADMIN values in shp file: {len(all_admin)}")

# 2. Unique country values after spatial join
joined_admin = set(result['country'].unique())
print(f"Number of unique country values after spatial join: {len(joined_admin)}")

# 3. Which countries/regions in shp are missing from result
missing_admin = all_admin - joined_admin
print(f"Countries/regions in shp but missing after spatial join: {list(missing_admin)}")

# 4. Cases where ISO3 is -99 or missing in result
print("Countries with ISO3 as -99 in result:", result.loc[result['iso3'] == '-99', 'country'].unique())
print("Number of missing ISO3 in result:", result['iso3'].isnull().sum())

# 5. Check the correspondence between country and iso3 in result
print("One-to-many relationship between country and iso3:")
print(result.groupby('iso3')['country'].nunique().value_counts())

Number of unique ADMIN values in shp file: 242
Number of unique country values after spatial join: 190
Countries/regions in shp but missing after spatial join: ['Heard Island and McDonald Islands', 'Dominica', 'Seychelles', 'British Indian Ocean Territory', 'Maldives', 'Federated States of Micronesia', 'Antigua and Barbuda', 'British Virgin Islands', 'Anguilla', 'Tonga', 'Guernsey', 'Wallis and Futuna', 'Saint Kitts and Nevis', 'Niue', 'Barbados', 'Pitcairn Islands', 'Liechtenstein', 'Saint Martin', 'Norfolk Island', 'Palau', 'Saint Lucia', 'Bermuda', 'Jersey', 'Curaçao', 'Grenada', 'Northern Mariana Islands', 'Tuvalu', 'American Samoa', 'Saint Barthelemy', 'Nauru', 'Singapore', 'Indian Ocean Territories', 'Aruba', 'Turks and Caicos Islands', 'Cook Islands', 'Vatican', 'Andorra', 'Saint Pierre and Miquelon', 'São Tomé and Principe', 'Guam', 'Montserrat', 'Hong Kong S.A.R.', 'Isle of Man', 'Ashmore and Cartier Islands', 'Saint Helena', 'Sint Maarten', 'Malta', 'Monaco', 'San Marino', 'M

In [12]:
# 2. Checking the area of polygons for countries that are missing after spatial connectivity
missing_countries_gdf = countries[countries['ADMIN'].isin(missing_admin)].copy()
missing_countries_gdf['area'] = missing_countries_gdf['geometry'].to_crs('EPSG:3857').area / 1e6  # 单位：平方公里
print(missing_countries_gdf[['ADMIN', 'area']].sort_values('area'))

# 3. Checking if there are SPEI grid points falling into these countries (with serial output)
for idx, admin in enumerate(missing_admin, 1):
    poly = missing_countries_gdf[missing_countries_gdf['ADMIN'] == admin].geometry.iloc[0]
    n_points = points_gdf.within(poly).sum()
    print(f"{idx}. SPEI grid points in {admin}: {n_points}")

# Get the names and ISO_A3 of missing countries
missing_countries_gdf = countries[countries['ADMIN'].isin(missing_admin)][['ADMIN', 'ISO_A3']]
missing_countries_gdf = missing_countries_gdf.reset_index(drop=True)
print(missing_countries_gdf)
missing_countries_gdf.to_csv('../data/processed/spei_missing_countries.csv', index=False)


                                 ADMIN         area
5                              Vatican     1.278113
229        Ashmore and Cartier Islands     2.856467
241                             Tuvalu    10.503433
108                             Monaco    23.847567
196                        Macao S.A.R    26.595111
164                   Saint Barthelemy    27.142060
100                              Nauru    27.859625
240                       Sint Maarten    46.381684
20                    Pitcairn Islands    46.713122
228                     Norfolk Island    53.958729
163                       Saint Martin    54.358330
114                           Maldives    66.847959
95                        Cook Islands    69.953413
27                          Montserrat    87.193406
21                            Anguilla    92.520471
24                             Bermuda    92.584186
29                            Guernsey   114.037389
14                      American Samoa   128.520951
69          

The following code solves the case of one-to-many between country and ISO_A3, i.e.: countries with ISO_A3 of -99 in result: ['France' 'Kosovo' 'Northern Cyprus' 'Norway' 'Siachen Glacier'
 'Somaliland'], of which Northern Cyprus is not recognised by the UN, so there is no iso_A3, we delete it here; the others are matched manually.

In [13]:
# Directly delete Northern Cyprus from result
result = result[result['country'] != 'Northern Cyprus'].copy()

In [14]:
# 1. Building mapping tables manually
iso3_manual_map = {
    'France': 'FRA',
    'Kosovo': 'XKX',
    'Norway': 'NOR',
    'Siachen Glacier': 'SIA',
    'Somaliland': 'SOL'
}

# 2. Replace ISO_A3 in spei_clean
def fix_iso3(row):
    if row['iso3'] == '-99':
        return iso3_manual_map.get(row['country'], None)
    else:
        return row['iso3']

result = result[result['country'] != 'Northern Cyprus'].copy()
result['iso3_fixed'] = result.apply(fix_iso3, axis=1)

# 3. Check the results of the correction
print(result[result['iso3'] == '-99'][['country', 'iso3', 'iso3_fixed']])

# 4. Replace the original column with the corrected iso3
result['iso3'] = result['iso3_fixed']
result = result.drop(columns=['iso3_fixed'])


               country iso3 iso3_fixed
2337            France  -99        FRA
2369            Kosovo  -99        XKX
2404            Norway  -99        NOR
2426   Siachen Glacier  -99        SIA
2432        Somaliland  -99        SOL
...                ...  ...        ...
11267           France  -99        FRA
11299           Kosovo  -99        XKX
11334           Norway  -99        NOR
11356  Siachen Glacier  -99        SIA
11362       Somaliland  -99        SOL

[240 rows x 3 columns]


In [15]:
# Check for new RESULTS, counting countries and iso3 unique quantities
print("After removing Northern Cyprus, unique country values in result:", result['country'].nunique())
print("After removing Northern Cyprus, unique iso3 values in result:", result['iso3'].nunique())

After removing Northern Cyprus, unique country values in result: 189
After removing Northern Cyprus, unique iso3 values in result: 189


Checks completed:
1. 52 countries above that are not connected because they are too small to have spei grid points to fall into;
2. fixed 5 ISO-3 wrong country codes (-99->real ISO-A3), removed Northern Cyprus which is not recognised by the UN
3. 242-52-1 = 189
3. the current RESULTS are all with at least one grid point falling into the country boundaries, but need to be checked again as the spei may have null values.



In [16]:
# Check for overall deficiencies
print("Total rows:", len(result))
print("Missing values per column:\n", result.isnull().sum())

# Check which countries are missing
missing_country = result[result['spei'].isnull()]
print("Number of countries with missing values:", missing_country['country'].nunique())
print("Countries with missing values:", missing_country['country'].unique())

Total rows: 9072
Missing values per column:
 country        0
iso3           0
spei         144
spei_date      0
year           0
month          0
spei_lag1    144
spei_lag2    144
spei_lag3    144
dtype: int64
Number of countries with missing values: 3
Countries with missing values: ['Antarctica' 'Cayman Islands' 'Kiribati']


In [17]:
# Check Kiribati
country_poly = countries[countries['ADMIN'] == 'Kiribati'].geometry.iloc[0]
points_in_country = points_gdf[points_gdf.within(country_poly)]
print(f"Number of grid points in Kiribati: {len(points_in_country)}")

# Find the longitude and latitude of the point
pt = points_in_country.geometry.iloc[0]
lon, lat = pt.x, pt.y

# Find the nearest SPEI grid index
lat_idx = np.argmin(np.abs(lats - lat))
lon_idx = np.argmin(np.abs(lons - lon))

# Check all time SPEI values
spei_values = spei[:, lat_idx, lon_idx].values
print("Are all SPEI values for this point NaN?:", np.all(np.isnan(spei_values)))
print("Some SPEI values:", spei_values[:10])

Number of grid points in Kiribati: 1
Are all SPEI values for this point NaN?: True
Some SPEI values: [nan nan nan nan nan nan nan nan nan nan]


In [18]:
# Check Cayman Islands
country_poly_CI = countries[countries['ADMIN'] == 'Cayman Islands'].geometry.iloc[0]
points_in_country_CI = points_gdf[points_gdf.within(country_poly_CI)]
print(f"Number of grid points in Cayman Islands: {len(points_in_country_CI)}")

# Find the longitude and latitude of the point
pt_CI = points_in_country_CI.geometry.iloc[0]
lon_CI, lat_CI = pt_CI.x, pt_CI.y

# Find the nearest SPEI grid index
lat_idx_CI = np.argmin(np.abs(lats - lat_CI))
lon_idx_CI = np.argmin(np.abs(lons - lon_CI))

# Check all time SPEI values
spei_values_CI = spei[:, lat_idx_CI, lon_idx_CI].values
print("Are all SPEI values for this point NaN?:", np.all(np.isnan(spei_values_CI)))
print("Some SPEI values:", spei_values_CI[:10])


Number of grid points in Cayman Islands: 1
Are all SPEI values for this point NaN?: True
Some SPEI values: [nan nan nan nan nan nan nan nan nan nan]


In [19]:
# Check Antarctica
country_poly_A = countries[countries['ADMIN'] == 'Antarctica'].geometry.iloc[0]
points_in_country_A = points_gdf[points_gdf.within(country_poly_A)]
print(f"Number of grid points in Antarctica: {len(points_in_country_A)}")

# Find the longitude and latitude of the point
pt_A = points_in_country_A.geometry.iloc[0]
lon_A, lat_A = pt_A.x, pt_A.y

# Find the nearest SPEI grid index
lat_idx_A = np.argmin(np.abs(lats - lat_A))
lon_idx_A = np.argmin(np.abs(lons - lon_A))

# Check all time SPEI values
spei_values_A = spei[:, lat_idx_A, lon_idx_A].values
print("Are all SPEI values for this point NaN?:", np.all(np.isnan(spei_values_A)))
print("Some SPEI values:", spei_values_A[:10])


Number of grid points in Antarctica: 24165
Are all SPEI values for this point NaN?: True
Some SPEI values: [nan nan nan nan nan nan nan nan nan nan]


So the result of the check is that all three places have spei grid points, but the values of spei are indeed all Nan

In [20]:
# Delete the three missing States
drop_countries = ['Antarctica', 'Cayman Islands', 'Kiribati']
spei_clean = result[~result['country'].isin(drop_countries)].copy()

# Check for any remaining missing values
print("Missing values per column:\n", spei_clean.isnull().sum())
print("Total rows:", len(spei_clean))

Missing values per column:
 country      0
iso3         0
spei         0
spei_date    0
year         0
month        0
spei_lag1    0
spei_lag2    0
spei_lag3    0
dtype: int64
Total rows: 8928


In [21]:
spei_clean

,country,iso3,spei,spei_date,year,month,spei_lag1,spei_lag2,spei_lag3
2280,Afghanistan,AFG,-0.101823,2019-01-16,2019,1,-0.092170,0.173375,-0.499876
2281,Aland,ALA,-0.584555,2019-01-16,2019,1,-0.591903,-0.573946,-0.568361
2282,Albania,ALB,-0.229678,2019-01-16,2019,1,-1.489391,-1.807115,-1.576336
2283,Algeria,DZA,-1.081419,2019-01-16,2019,1,-0.542777,0.313114,0.784864
2284,Angola,AGO,0.090749,2019-01-16,2019,1,0.083615,0.132651,0.015039
...,...,...,...,...,...,...,...,...,...
11395,Western Sahara,ESH,-1.458937,2022-12-16,2022,12,-0.939643,-0.582081,-0.597001
11396,Yemen,YEM,-1.207492,2022-12-16,2022,12,-0.505910,0.006843,0.205940
11397,Zambia,ZMB,0.084212,2022-12-16,2022,12,0.138420,0.039404,-0.045226
11398,Zimbabwe,ZWE,-0.164138,2022-12-16,2022,12,-0.331173,-0.593602,-0.502948


Now that the data cleaning is complete, the resulting spei_country_month_2019_2022_cleaned.csv is free of missing values and the ISO3 is correct and complete

In [22]:
# Final data check

print("Time range:", spei_clean['spei_date'].min(), "to", spei_clean['spei_date'].max())
print("Are there any missing values in the 'spei' column?:", spei_clean['spei'].isnull().sum() == 0)
print("Are there any missing values in the 'iso3' column?:", spei_clean['iso3'].isnull().sum() == 0)
print("Are there any '-99' values in the 'iso3' column?:", (spei_clean['iso3'] == '-99').sum() == 0)
print("Number of countries:", spei_clean['country'].nunique())
print("Number of iso3 codes:", spei_clean['iso3'].nunique())

# Count of SPEI values per country
spei_count = spei_clean.groupby('country')['spei'].count()
print("Are all SPEI values per country 48?:", (spei_count == 48).all())
print("Countries with SPEI values not equal to 48:")
print(spei_count[spei_count != 48])

Time range: 2019-01-16 00:00:00 to 2022-12-16 00:00:00
Are there any missing values in the 'spei' column?: True
Are there any missing values in the 'iso3' column?: True
Are there any '-99' values in the 'iso3' column?: True
Number of countries: 186
Number of iso3 codes: 186
Are all SPEI values per country 48?: True
Countries with SPEI values not equal to 48:
Series([], Name: spei, dtype: int64)


In [23]:
# === Modify and save data ===
# Drop the country column and rename iso3 to spei_iso3
spei_clean = spei_clean.drop(columns=['country'])
spei_clean = spei_clean.rename(columns={'iso3': 'spei_iso3'})

# Save the modified data
spei_clean.to_csv('../data/processed/spei_clean.csv', index=False)
print("Final panel saved, no missing values, 48 periods per country, ISO3 renamed to spei_iso3, saved as spei_clean.csv")

Final panel saved, no missing values, 48 periods per country, ISO3 renamed to spei_iso3, saved as spei_clean.csv


In [24]:
spei_clean

,spei_iso3,spei,spei_date,year,month,spei_lag1,spei_lag2,spei_lag3
2280,AFG,-0.101823,2019-01-16,2019,1,-0.092170,0.173375,-0.499876
2281,ALA,-0.584555,2019-01-16,2019,1,-0.591903,-0.573946,-0.568361
2282,ALB,-0.229678,2019-01-16,2019,1,-1.489391,-1.807115,-1.576336
2283,DZA,-1.081419,2019-01-16,2019,1,-0.542777,0.313114,0.784864
2284,AGO,0.090749,2019-01-16,2019,1,0.083615,0.132651,0.015039
...,...,...,...,...,...,...,...,...
11395,ESH,-1.458937,2022-12-16,2022,12,-0.939643,-0.582081,-0.597001
11396,YEM,-1.207492,2022-12-16,2022,12,-0.505910,0.006843,0.205940
11397,ZMB,0.084212,2022-12-16,2022,12,0.138420,0.039404,-0.045226
11398,ZWE,-0.164138,2022-12-16,2022,12,-0.331173,-0.593602,-0.502948


The above are all valid codes